In [1]:
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
load_dotenv()

True

In [2]:
data = """PERSONAL LOAN POLICY — BAJAJ FINANCE
Eligibility: Age 21-60 years. Minimum income Rs 25,000 per month.
CIBIL score 700 and above for standard approval.
FOIR maximum 50 percent of monthly income.
Interest Rates:
CIBIL 750+: 11 to 13 percent per annum
CIBIL 700-749: 13 to 16 percent per annum
CIBIL below 650: Not eligible
EMI bounce charge: Rs 1000 plus GST per bounce.
Late payment penalty: 2 percent per month on overdue amount.
GRIEVANCE REDRESSAL POLICY — BAJAJ FINANCE
Level 1: Helpdesk 1800-103-3535, 8AM to 8PM Mon-Sat. Resolution: 3 working days.
Level 2: Nodal officer email. Resolution: 7 working days.
Level 3: RBI Banking Ombudsman after 30 days unresolved.
NOC issuance: 7 working days after full loan closure.
Loan statement: Available instantly on portal.
Interest certificate: 2 working days from branch.
GOLD LOAN POLICY — BAJAJ FINANCE
Gold purity accepted: 18 karat to 24 karat.
Maximum LTV: 75 percent of market value (RBI mandated).
Tenure: 3 months to 24 months.
Foreclosure Policy:
Within 3 months: 2 percent foreclosure charge
3 to 6 months: 2 percent charge
6 to 12 months: 1 percent charge
After 12 months: No foreclosure charge
Disbursal: Same day within 30 minutes of appraisal.
# Add more things about Home loan
To take the home loan below is the eligibility criteria
1. Age: Minimum 21 years and maximum 65 years.
2. Income: Minimum net monthly income of Rs. 30,000.
3. Employment: Salaried or self-employed with a stable income source.
4. Credit Score: Minimum CIBIL score of 750.
5. Property Documents: Clear title and ownership documents.
# Leave pending for Rahul
Rahul has 5 days of leave remaining.
"""

In [19]:
# lets create the chunks from data
chunks = data.split("\n")
vocab = set()
for chunk in chunks:
    words = chunk.split()
    vocab.update(words)
# print(vocab)
# print(len(vocab))


def bow_vectorize(text):
    words = text.split()
    vector = [1 if word in words else 0 for word in vocab]
    return vector


store = [bow_vectorize(chunk) for chunk in chunks]

In [63]:
# questions = "what is interest rate for personal loan"
# questions_vector = bow_vectorize(questions)

# # compute the distance
# distances=[]
# for i, vector in enumerate(store):
#     distance = sum((a - b) ** 2 for a, b in zip(vector, questions_vector)) ** 0.5
#     distances.append((i, distance))

# distances


def result(questions,top=3):
    questions_vector = bow_vectorize(questions)

    # compute the distance
    distances=[]
    for i, vector in enumerate(store):
        distance = sum((a - b) ** 2 for a, b in zip(vector, questions_vector)) ** 0.5
        distances.append((i, distance))

    # sort distances
    distances.sort(key=lambda x: x[1])

    # get top n contexts
    context = [chunks[i] for i, _ in distances[:top]]
    text = "".join(context)

    # return top n contexts
    return text

In [64]:
result("personal loan")

'Interest Rates:Foreclosure Policy:'

In [50]:
# chunks[4]

In [51]:
llm = ChatOpenAI(model="gpt-4o-mini")


In [ ]:
template = ChatPromptTemplate.from_template("""
You are BajajBot, an AI assistant for Bajaj Finance helpdesk agents.

CONTEXT :
-------------------------------------------------------
{context}
-------------------------------------------------------

INSTRUCTIONS:
- Answer ONLY using the CONTEXT above.
- Do NOT use your training knowledge.
- If the answer is not in the CONTEXT, say exactly:
  "I don't have this information in the provided documents."
- Keep your answer under 3 sentences.
- If a specific number or rule is in CONTEXT, include it.

QUESTION: {question}

ANSWER:
""")



In [61]:
chain =  template | llm | StrOutputParser()

In [65]:
questions = "what is interest rate for personal loan"

context = result(questions)

chain.invoke({"context": context, "question": questions})


"I don't have this information in the provided documents."

In [66]:
context

'Interest Rates:Foreclosure Policy:'